# Basis Transforms

`nrpy.equations.basis_transforms.jacobians` applies the Jacobian and inverse-Jacobian matrices already stored on a `ReferenceMetric`. It does not rebuild the coordinate map; it reuses the selected reference metric as the transformation source.

## Table of Contents

1. [Imports and reference metric](#Imports-and-reference-metric)
2. [Radial vector to Cartesian components](#Radial-vector-to-Cartesian-components)
3. [Contravariant vector round trip](#Contravariant-vector-round-trip)
4. [Rank-2 covariant tensor example](#Rank-2-covariant-tensor-example)
5. [Next notebooks](#Next-notebooks)

## Imports and Reference Metric

The spherical reference metric supplies the coordinate map and Jacobians. `BasisTransforms("Spherical")` retrieves that metric from the current reference-metric cache.

In [ ]:
import nrpy
import nrpy.indexedexp as ixp
import nrpy.params as par
import nrpy.reference_metric as refmetric
import sympy as sp
from nrpy.equations.basis_transforms.jacobians import BasisTransforms


print("Imported nrpy from:", nrpy.__file__)

spherical = refmetric.reference_metric["Spherical"]
transforms = BasisTransforms("Spherical")
print("Reference metric:", spherical.CoordSystem)
print("Scale factors:", spherical.scalefactor_orthog)
print("Parallelization parameter:", par.parval_from_str("parallelization"))

## Radial Vector to Cartesian Components

A purely radial contravariant vector in the reference-metric basis becomes the familiar Cartesian radial direction.

In [ ]:
radial_amplitude = sp.Symbol("V_r", real=True)
radial_vectorU = [radial_amplitude, sp.sympify(0), sp.sympify(0)]
cartesian_vectorU = transforms.basis_transform_vectorU_from_rfmbasis_to_Cartesian(
    radial_vectorU
)

print("Cartesian components:")
for component in cartesian_vectorU:
    print(" ", sp.trigsimp(component))

## Contravariant Vector Round Trip

Transforming a symbolic contravariant vector from the reference-metric basis to Cartesian and back should return the original components. Zero residuals show that the stored Jacobians are mutually consistent for this basis operation.

In [ ]:
generic_vectorU = ixp.declarerank1("V")
cartesian_genericU = transforms.basis_transform_vectorU_from_rfmbasis_to_Cartesian(
    generic_vectorU
)
round_trip_vectorU = transforms.basis_transform_vectorU_from_Cartesian_to_rfmbasis(
    cartesian_genericU
)
round_trip_residuals = [
    sp.trigsimp(sp.simplify(round_trip_vectorU[i] - generic_vectorU[i]))
    for i in range(3)
]

print("Contravariant vector residuals:", round_trip_residuals)

## Rank-2 Covariant Tensor Example

The spherical reference metric tensor transformed to the Cartesian basis gives the Cartesian identity metric. The selected residuals below keep the output compact.

In [ ]:
cartesian_metricDD = transforms.basis_transform_tensorDD_from_rfmbasis_to_Cartesian(
    transforms.rfm.ghatDD
)
diagonal_residuals = [
    sp.trigsimp(sp.simplify(cartesian_metricDD[i][i] - 1)) for i in range(3)
]
off_diagonal_residual = sp.trigsimp(sp.simplify(cartesian_metricDD[0][1]))

print("Cartesian metric diagonal residuals:", diagonal_residuals)
print("Cartesian metric (0, 1) residual:", off_diagonal_residual)

## Next Notebooks

[Curvilinear wave equation](../3-wave_equation/wave_equation_curvilinear.ipynb) uses the same reference-metric data in a PDE right-hand side. Later numerical-relativity notebooks reuse these transformations for BSSN variables, initial data, and stress-energy tensors.